In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

from sklearn.feature_selection import VarianceThreshold

from sklearn.model_selection import (
    train_test_split,
    cross_val_score,
    StratifiedKFold,
    GridSearchCV
)

from sklearn.preprocessing import (
    LabelEncoder,
    StandardScaler
)

from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression

from sklearn.tree import DecisionTreeClassifier

from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

In [ ]:
# ΚΕΛΙ 2
folder_path = "MachineLearningCSV"

files = [f for f in os.listdir(folder_path) if f.endswith('.csv')]

df_list = []

for file in files:
    temp_df = pd.read_csv(os.path.join(folder_path, file))
    df_list.append(temp_df)

df = pd.concat(df_list, ignore_index=True)
df.columns = df.columns.str.strip()

print("Dataset loaded successfully!")

In [ ]:
# ΚΕΛΙ 3 
print("Dataset Shape:")
print(df.shape)

print("\nColumn Names:")
print(df.columns.tolist())

print("\nData Types:")
print(df.dtypes)

print("\nDataset Info:")
df.info()

In [ ]:
# ΚΕΛΙ 4
feature_description = {
    "Flow Duration": "Duration of the flow in microseconds",
    "Total Fwd Packets": "Total packets in forward direction",
    "Total Backward Packets": "Total packets in backward direction",
    "Flow Bytes/s": "Bytes transferred per second",
    "Flow Packets/s": "Packets transferred per second",
    "Label": "Traffic category / attack type"
}

for key, value in feature_description.items():
    print(f"{key} --> {value}")

In [ ]:
# ΚΕΛΙ 5 

print("Initial Shape:", df.shape)

print("\nMissing Values Per Column:")
print(df.isnull().sum())

print("\nTotal Missing Values:")
print(df.isnull().sum().sum())

print("\nDuplicate Rows:")
print(df.duplicated().sum())

In [ ]:
# ΚΕΛΙ 6
# Remove duplicates
df.drop_duplicates(inplace=True)

# Replace inf values with NaN
df.replace([np.inf, -np.inf], np.nan, inplace=True)

# Remove missing values
df.dropna(inplace=True)

# Αφαιρεί οτιδήποτε ΔΕΝ είναι γράμμα, αριθμός ή κενό
df['Label'] = df['Label'].str.replace(r'[^\w\s]', '', regex=True).str.strip()

print("Final Shape After Cleaning:", df.shape)

In [ ]:
# ΚΕΛΙ 7
stats = df.describe().T

print(stats)

In [ ]:
# ΚΕΛΙ 8
print(df['Label'].value_counts())

print("\nPercentages:")
print(df['Label'].value_counts(normalize=True) * 100)

plt.figure(figsize=(14,6))

sns.countplot(
    y=df['Label'],
    order=df['Label'].value_counts().index
)

plt.xscale('log')

plt.title("Label Distribution")
plt.xlabel("Count (log scale)")
plt.ylabel("Attack Type")

plt.show()

In [ ]:
# ΚΕΛΙ 9: Επιλογή Χαρακτηριστικών (Feature Selection)

numeric_df = df.select_dtypes(include=np.number)

corr_matrix = numeric_df.corr()

plt.figure(figsize=(18,14))

sns.heatmap(
    corr_matrix,
    cmap='coolwarm',
    center=0
)

plt.title("Correlation Heatmap")

plt.show()

In [ ]:
# ΚΕΛΙ 10
upper = corr_matrix.where(
    np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
)

high_corr_features = [
    column for column in upper.columns
    if any(upper[column] > 0.95)
]

print("Highly Correlated Features:")
print(high_corr_features)

In [ ]:
#ΚΕΛΙ 11
numeric_df = df.select_dtypes(include=np.number)

# Remove NaN rows
numeric_df.dropna(inplace=True)

In [ ]:
#ΚΕΛΙ 12
selector = VarianceThreshold(threshold=0.01)

selector.fit(numeric_df)

low_variance_features = numeric_df.columns[
    ~selector.get_support()
]

print("Low Variance Features:")
print(list(low_variance_features))

In [ ]:
# ΚΕΛΙ 13
features_to_drop = list(high_corr_features) + list(low_variance_features)

features_to_drop = list(set(features_to_drop))

print("Features to remove:")
print(features_to_drop)

df_reduced = df.drop(columns=features_to_drop)

print("\nOriginal Shape:", df.shape)
print("Reduced Shape:", df_reduced.shape)

In [ ]:
# ΚΕΛΙ 14
numeric_df.hist(
    figsize=(20,20),
    bins=30
)

plt.tight_layout()
plt.show()

In [ ]:
# ΚΕΛΙ 15
sample_columns = numeric_df.columns[:10]

for col in sample_columns:

    plt.figure(figsize=(10,4))

    sns.boxplot(x=numeric_df[col])

    plt.title(f"Boxplot of {col}")

    plt.show()

In [ ]:
# ΚΕΛΙ 16
sns.scatterplot(
    data=df,
    x='Flow Duration',
    y='Flow Bytes/s',
    hue='Label'
)

plt.title("Flow Duration vs Flow Bytes/s")

plt.show()

In [ ]:
# ΚΕΛΙ 17
# Features
X = df_reduced.drop(columns=['Label'])

# Target
y = df_reduced['Label']

print("Features Shape:", X.shape)
print("Target Shape:", y.shape)

In [ ]:
# ΚΕΛΙ 18
# Τα ML μοντέλα θέλουν αριθμούς
label_encoder = LabelEncoder()

y_encoded = label_encoder.fit_transform(y)

print("Classes:")
print(label_encoder.classes_)

In [ ]:
# ΚΕΛΙ 19
# Πρώτο split: train+validation και test
X_temp, X_test, y_temp, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

# Δεύτερο split: train και validation
X_train, X_val, y_train, y_val = train_test_split(
    X_temp,
    y_temp,
    test_size=0.25,
    random_state=42,
    stratify=y_temp
)

print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

In [ ]:
# ΚΕΛΙ 20
# Feature Scaling
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_val_scaled = scaler.transform(X_val)

In [ ]:
# ΚΕΛΙ 21
# Logistic Regression

log_reg = LogisticRegression(
    max_iter=2000,
    class_weight='balanced',
    random_state=42
)

log_reg.fit(X_train_scaled, y_train)

y_pred_log = log_reg.predict(X_test_scaled)

print("Accuracy:")
print(accuracy_score(y_test, y_pred_log))

print("\nPrecision:")
print(precision_score(y_test, y_pred_log, average='weighted'))

print("\nRecall:")
print(recall_score(y_test, y_pred_log, average='weighted'))

print("\nF1-score:")
print(f1_score(y_test, y_pred_log, average='weighted'))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_log))

In [ ]:
# ΚΕΛΙ 22
# Decision Tree

decision_tree = DecisionTreeClassifier(
    class_weight='balanced',
    random_state=42
)

decision_tree.fit(X_train, y_train)

y_pred_tree = decision_tree.predict(X_test)

print("Accuracy:")
print(accuracy_score(y_test, y_pred_tree))

print("\nPrecision:")
print(precision_score(y_test, y_pred_tree, average='weighted'))

print("\nRecall:")
print(recall_score(y_test, y_pred_tree, average='weighted'))

print("\nF1-score:")
print(f1_score(y_test, y_pred_tree, average='weighted'))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_tree))

In [ ]:
# ΚΕΛΙ 23
# Random Forest

random_forest = RandomForestClassifier(
    class_weight='balanced',
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

random_forest.fit(X_train, y_train)

y_pred_rf = random_forest.predict(X_test)

print("Accuracy:")
print(accuracy_score(y_test, y_pred_rf))

print("\nPrecision:")
print(precision_score(y_test, y_pred_rf, average='weighted'))

print("\nRecall:")
print(recall_score(y_test, y_pred_rf, average='weighted'))

print("\nF1-score:")
print(f1_score(y_test, y_pred_rf, average='weighted'))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_rf))

C: ελέγχει το regularization.
Μικρές τιμές → πιο αυστηρό regularization.
Μεγάλες τιμές → πιο flexible model.

In [ ]:
# ΚΕΛΙ 24
# Manual Grid Search - Logistic Regression

param_grid_lr = {
    'C': [0.01, 0.1, 1, 10]
}

best_f1 = 0
best_params_lr = None
best_model_lr = None

for C in param_grid_lr['C']:

    model = LogisticRegression(
        C=C,
        solver='liblinear',
        penalty='l2',
        max_iter=2000,
        class_weight='balanced',
        random_state=42
    )

    model.fit(X_train_scaled, y_train)

    val_pred = model.predict(X_val_scaled)

    score = f1_score(
        y_val,
        val_pred,
        average='weighted'
    )

    print(f"C={C} -> F1={score:.4f}")

    if score > best_f1:
        best_f1 = score
        best_params_lr = C
        best_model_lr = model

print("\nBest Parameter:")
print(best_params_lr)

print("Best Validation F1:")
print(best_f1)

In [ ]:
# ΚΕΛΙ 24.5
# Evaluation on Test Set

y_test_pred_lr = best_model_lr.predict(X_test_scaled)

print(classification_report(y_test, y_test_pred_lr))

print("Accuracy:")
print(accuracy_score(y_test, y_test_pred_lr))

print("Precision:")
print(precision_score(y_test, y_test_pred_lr, average='weighted'))

print("Recall:")
print(recall_score(y_test, y_test_pred_lr, average='weighted'))

print("F1:")
print(f1_score(y_test, y_test_pred_lr, average='weighted'))

In [ ]:
# ΚΕΛΙ 25
# Confusion Matrix for Logistic Regression
cm = confusion_matrix(y_test, y_pred_log)

plt.figure(figsize=(10,8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Logistic Regression Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

max_depth: έλεγχος overfitting.
min_samples_split: αποφυγή πολύ μικρών splits.
criterion: τρόπος υπολογισμού impurity.

In [ ]:
# ΚΕΛΙ 25
# Manual Grid Search - Decision Tree

param_grid_dt = {
    'max_depth': [5, 10, 20],
    'min_samples_split': [2, 5, 10],
    'criterion': ['gini', 'entropy']
}

best_f1 = 0
best_params_dt = None
best_model_dt = None

for depth in param_grid_dt['max_depth']:
    for split in param_grid_dt['min_samples_split']:
        for criterion in param_grid_dt['criterion']:

            model = DecisionTreeClassifier(
                max_depth=depth,
                min_samples_split=split,
                criterion=criterion,
                class_weight='balanced',
                random_state=42
            )

            model.fit(X_train, y_train)

            val_pred = model.predict(X_val)

            score = f1_score(
                y_val,
                val_pred,
                average='weighted'
            )

            print(depth, split, criterion, score)

            if score > best_f1:
                best_f1 = score
                best_params_dt = {
                    'max_depth': depth,
                    'min_samples_split': split,
                    'criterion': criterion
                }
                best_model_dt = model

print(best_params_dt)
print(best_f1)

In [ ]:
# ΚΕΛΙ 25.5
# Test Evaluation - Decision Tree

y_test_pred_dt = best_model_dt.predict(X_test)

print(classification_report(y_test, y_test_pred_dt))

print("Accuracy:")
print(accuracy_score(y_test, y_test_pred_dt))

print("Precision:")
print(precision_score(y_test, y_test_pred_dt,
                      average='weighted'))

print("Recall:")
print(recall_score(y_test, y_test_pred_dt,
                   average='weighted'))

print("F1:")
print(f1_score(y_test, y_test_pred_dt,
               average='weighted'))

n_estimators: αριθμός δέντρων.
Περισσότερα trees → πιο σταθερό ensemble.
max_depth: έλεγχος πολυπλοκότητας.

In [ ]:
# ΚΕΛΙ 26
# Μanual Grid Search - Random Forest
param_grid_rf = {
    'n_estimators': [50, 100],
    'max_depth': [10, 20],
    'min_samples_split': [2, 5]
}

best_score = 0
best_params_rf = None
best_model_rf = None

for n in param_grid_rf['n_estimators']:
    for depth in param_grid_rf['max_depth']:
        for split in param_grid_rf['min_samples_split']:

            model = RandomForestClassifier(
                n_estimators=n,
                max_depth=depth,
                min_samples_split=split,
                class_weight='balanced',
                random_state=42,
                n_jobs=-1
            )

            model.fit(X_train, y_train)

            val_pred = model.predict(X_val)

            score = f1_score(
                y_val,
                val_pred,
                average='weighted'
            )

            print(n, depth, split, score)

            if score > best_score:
                best_score = score
                best_params_rf = {
                    'n_estimators': n,
                    'max_depth': depth,
                    'min_samples_split': split
                }
                best_model_rf = model

print(best_params_rf)
print(best_score)

In [ ]:
# ΚΕΛΙ 26.5
y_test_pred_rf = best_model_rf.predict(X_test)

print(classification_report(y_test, y_test_pred_rf))

print("Accuracy:",
      accuracy_score(y_test, y_test_pred_rf))

print("Precision:",
      precision_score(y_test, y_test_pred_rf,
                      average='weighted'))

print("Recall:",
      recall_score(y_test, y_test_pred_rf,
                   average='weighted'))

print("F1:",
      f1_score(y_test, y_test_pred_rf,
               average='weighted'))

Ο αριθμός estimators επηρεάζει τη σταθερότητα του ensemble.

In [ ]:
# ΚΕΛΙ 28
# Grid Search CV for Logistic Regression using Pipeline

lr_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(
        max_iter=2000,
        class_weight='balanced',
        random_state=42
    ))
])

# Hyperparameters for Pipeline
param_grid_lr_cv = {
    'model__C': [0.01, 0.1, 1, 10],
    'model__penalty': ['l2'],
    'model__solver': ['liblinear']
}

# Grid Search
grid_search_lr = GridSearchCV(
    lr_pipeline,
    param_grid_lr_cv,
    cv=StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=42
    ),
    scoring='f1_weighted',
    n_jobs=-1
)

# εδώ βάζουμε X_train ΟΧΙ X_train_scaled
grid_search_lr.fit(X_train, y_train)

print("Best Params:")
print(grid_search_lr.best_params_)

print("\nBest CV Score:")
print(grid_search_lr.best_score_)

Επιλέχθηκε το weighted F1-score λόγω ανισορροπίας των κλάσεων.

In [ ]:
# ΚΕΛΙ 29
# Grid Search using sklearn's GridSearchCV for Decision Tree
grid_search_dt = GridSearchCV(
    DecisionTreeClassifier(class_weight='balanced',random_state=42),
    param_grid_dt,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring='f1_weighted',
    n_jobs=-1
)

grid_search_dt.fit(X_train, y_train)

print(grid_search_dt.best_params_)
print(grid_search_dt.best_score_)

In [ ]:
# ΚΕΛΙ 30
#  Confusion Matrix - Decision Tree

cm = confusion_matrix(y_test, y_test_pred_dt)

plt.figure(figsize=(10,8))

sns.heatmap(cm,
            annot=True,
            fmt='d',
            cmap='Greens')

plt.title('Decision Tree Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

Επιλέχθηκε το weighted F1-score λόγω ανισορροπίας των κλάσεων.

In [ ]:
# ΚΕΛΙ 31
# Grid Search using sklearn's GridSearchCV for Random Forest

grid_search_rf = GridSearchCV(
    RandomForestClassifier(
        class_weight='balanced',
        random_state=42,
        n_jobs=-1
    ),

    param_grid_rf,

    cv=StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=42
    ),

    scoring='f1_weighted',
    n_jobs=-1
)

grid_search_rf.fit(X_train, y_train)

print(grid_search_rf.best_params_)
print(grid_search_rf.best_score_)

In [ ]:
# Confusion Matrix - Random Forest

cm = confusion_matrix(y_test, y_test_pred_rf)

plt.figure(figsize=(10,8))

sns.heatmap(cm,
            annot=True,
            fmt='d',
            cmap='Oranges')

plt.title('Random Forest Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

Επιλέχθηκε το weighted F1-score λόγω ανισορροπίας των κλάσεων.

In [ ]:
# ΚΕΛΙ 32
# Cross-validation για όλα τα μοντέλα

models = {
    'Logistic Regression': Pipeline([
        ('scaler', StandardScaler()),
        ('model', LogisticRegression(
            max_iter=2000,
            class_weight='balanced',
            random_state=42
        ))
    ]),

    'Decision Tree': DecisionTreeClassifier(
        class_weight='balanced',
        random_state=42
    ),

    'Random Forest': RandomForestClassifier(
        class_weight='balanced',
        random_state=42
    )
}

for name, model in models.items():

    scores = cross_val_score(
        model,
        X,
        y_encoded,
        cv=StratifiedKFold(
            n_splits=5,
            shuffle=True,
            random_state=42
        ),
        scoring='f1_weighted',
        n_jobs=-1
    )

    print(f"\n{name}")
    print(f"Mean F1-score: {scores.mean():.4f}")

In [ ]:
# ΚΕΛΙ 33
# Evaluation - Logistic Regression CV

best_lr_cv = grid_search_lr.best_estimator_

lr_cv_pred = best_lr_cv.predict(X_test)

print(classification_report(y_test, lr_cv_pred))

print("Accuracy:")
print(accuracy_score(y_test, lr_cv_pred))

print("Precision:")
print(precision_score(y_test, lr_cv_pred,
                      average='weighted'))

print("Recall:")
print(recall_score(y_test, lr_cv_pred,
                   average='weighted'))

print("F1:")
print(f1_score(y_test, lr_cv_pred,
               average='weighted'))

Επιλέχθηκε το weighted F1-score λόγω ανισορροπίας των κλάσεων.

In [ ]:
# ΚΕΛΙ 34
# Evaluation - Decision Tree CV

best_dt_cv = grid_search_dt.best_estimator_

dt_cv_pred = best_dt_cv.predict(X_test)

print(classification_report(y_test, dt_cv_pred))

print("Accuracy:")
print(accuracy_score(y_test, dt_cv_pred))

print("Precision:")
print(precision_score(y_test, dt_cv_pred,
                      average='weighted'))

print("Recall:")
print(recall_score(y_test, dt_cv_pred,
                   average='weighted'))

print("F1:")
print(f1_score(y_test, dt_cv_pred,
               average='weighted'))

In [ ]:
# ΚΕΛΙ 35
# Evaluation - Random Forest CV

best_rf_cv = grid_search_rf.best_estimator_

rf_cv_pred = best_rf_cv.predict(X_test)

print(classification_report(y_test, rf_cv_pred))

print("Accuracy:")
print(accuracy_score(y_test, rf_cv_pred))

print("Precision:")
print(precision_score(y_test, rf_cv_pred,
                      average='weighted'))

print("Recall:")
print(recall_score(y_test, rf_cv_pred,
                   average='weighted'))

print("F1:")
print(f1_score(y_test, rf_cv_pred,
               average='weighted'))

In [ ]:
# ΚΕΛΙ 36
#  Final Comparison Table

results = pd.DataFrame({
    'Model': [
        'Logistic Regression',
        'Decision Tree',
        'Random Forest'
    ],

    'F1 Manual Grid Search': [
        f1_score(y_test, y_test_pred_lr, average='weighted'),
        f1_score(y_test, y_test_pred_dt, average='weighted'),
        f1_score(y_test, y_test_pred_rf, average='weighted')
    ],

    'F1 GridSearchCV': [
        f1_score(y_test, lr_cv_pred, average='weighted'),
        f1_score(y_test, dt_cv_pred, average='weighted'),
        f1_score(y_test, rf_cv_pred, average='weighted')
    ]
})

print(results)

In [ ]:
# ΚΕΛΙ 37
#  Visualization of Final Results

results.plot(
    x='Model',
    kind='bar',
    figsize=(10,6)
)

plt.title('Model Comparison')
plt.ylabel('Weighted F1-score')

plt.xticks(rotation=0)

plt.grid(axis='y')

plt.show()

Το Random Forest είχε καλύτερη συνολική επίδοση.
Το Cross Validation έδωσε πιο σταθερές hyperparameters.
Το train/validation/test split είναι πιο γρήγορο αλλά λιγότερο αξιόπιστο.
Το weighted F1-score επιλέχθηκε λόγω class imbalance.
Η Logistic Regression επηρεάστηκε περισσότερο από scaling.
Τα tree-based models δεν χρειάζονται scaling.